In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain.mcp import MCPAdapter
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field


In [ ]:
load_dotenv()

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [ ]:
class Classification(BaseModel):
    category: Literal["career", "relationships", "education", "health", "general"] = (
        Field(description="Defines category field of question asked by user")
    )

In [ ]:
class ChatState(BaseModel):
    # The current user request; kept separate from tool-call messages.
    question: HumanMessage

    messages: Annotated[list[AnyMessage], add_messages] = Field(
        default_factory=list,
        description="Research-agent and tool-call messages for the current question",
    )
    category: str | None = "Unclassified"
    skill_instructions: str | None = "None"

    # Snapshot loaded from the authoritative user-profile database.
    user_id: str
    birth_profile_version: int
    dob: str
    birth_time: str
    birth_place: str

    research_findings: str | None = Field(
        default=None, description="Verified summary for the final chat node"
    )
    final_answer: str | None = Field(
        default=None, description="Final polished output to the user"
    )

In [ ]:
classifier_llm = model.with_structured_output(Classification)

CLASSIFIER_SYSTEM_PROMPT = """
    Classify the user's astrology question into exactly one category.

    Categories:

    career:
    Questions about jobs, career growth, promotion,
    business, professional success and career timing.

    relationships:
    Questions about marriage, love, spouse,
    compatibility and family relationships.

    education:
    Questions about studies, academic performance,
    higher education and educational prospects.

    health:
    Questions about health, illness and health-related
    astrological indications.

    general:
    Questions about birth charts, planetary positions,
    dashas, transits, yogas, doshas, astrology concepts,
    or anything that does not clearly belong to the other categories.

    If multiple categories are involved, select the primary
    subject of the user's question. Question: {question} """


def classify_node(state: ChatState):
    question = state.question.content
    print("Question", question)
    result = classifier_llm.invoke(CLASSIFIER_SYSTEM_PROMPT.format(question))

    return {"category": result.category}

In [ ]:
from aistrology_langgraph_skeleton.skills.registry import load_skill


def load_skill_node(state: ChatState):
    skill = load_skill(state.category)
    return {"skill_instructions": skill}

In [ ]:
mcpConfig = {
    "mcpServers": {
        "vedic-astrology": {
            "type": "sse",
            "url": "https://jagannatha-hora-359167915530.europe-west1.run.app/mcp",
            "headers": {"x-api-key": "demo"},
        }
    }
}

# Getting Tools
async with MCPAdapter(mcpConfig) as adapter:
    tools = await adapter.list_tools()
    # print(tools)

In [ ]:
RESEARCH_SYSTEM_PROMPT = """
    You are AIstrology's research agent.

    Follow the domain-specific instructions below.

    {skill}

    User context:
    {user_context}

    Use tools when necessary to obtain accurate
    astrological calculations or retrieve knowledge.

    Return verified findings for the final-answer agent, rather than addressing
    the user directly.

    Never invent birth-chart calculations.
    """
llm_with_tools = model.bind_tools(tools)


def format_user_context(state: ChatState) -> str:
    return (
        f"user_id={state.user_id}\n"
        f"birth_profile_version={state.birth_profile_version}\n"
        f"dob={state.dob}\n"
        f"birth_time={state.birth_time}\n"
        f"birth_place={state.birth_place}"
    )


async def research_node(state: ChatState):
    prompt = RESEARCH_SYSTEM_PROMPT.format(
        skill=state.skill_instructions,
        user_context=format_user_context(state),
    )
    messages = [SystemMessage(content=prompt), state.question, *state.messages]

    response = await llm_with_tools.ainvoke(messages)

    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
def content_as_text(message: AnyMessage) -> str:
    if isinstance(message.content, str):
        return message.content
    if isinstance(message.content, list):
        return "\n".join(
            block["text"]
            for block in message.content
            if isinstance(block, dict)
            and block.get("type") == "text"
            and isinstance(block.get("text"), str)
        )
    return ""


def summarize_findings_node(state: ChatState):
    transcript = "\n\n".join(
        f"{message.type}: {content_as_text(message)}"
        for message in state.messages
        if content_as_text(message).strip()
    )
    # 1. Keep instructions in the system message
    system_instruction = (
        "Summarize this research for a final chat agent. Use only verified facts."
    )

    # 2. Provide the actual content inside a HumanMessage
    user_content = (
        f"Research transcript:\n{transcript or 'No usable research was returned.'}"
    )

    summary = content_as_text(
        model.invoke(
            [
                SystemMessage(content=system_instruction),
                HumanMessage(content=user_content),
            ]
        )
    ).strip()
    return {"research_findings": summary}


def chat_node(state: ChatState):
    SYSTEM_PROMPT = """You are Aistrology, a helpful and thoughtful chat assistant.
    Use the following user context when answering astrology questions:
    {user_context}

    Use these verified research findings as source material:
    {research_findings}
    Understand the user's request, reason carefully in private, and give a clear,
    direct final answer. Do not reveal hidden chain-of-thought or private reasoning.
    If information is uncertain, say so plainly."""

    prompt = SYSTEM_PROMPT.format(
        user_context=format_user_context(state),
        research_findings=state.research_findings
        or "No verified findings were available.",
    )
    response = model.invoke([SystemMessage(content=prompt), state.question])

    return {"final_answer": content_as_text(response).strip() or None}


In [ ]:
# Graph
graph = StateGraph(ChatState)
graph.add_node("chat", chat_node)
graph.add_node("classify", classify_node)
graph.add_node("load_skill", load_skill_node)
graph.add_node("tools", tool_node)
graph.add_node("research", research_node)
graph.add_node("summarize", summarize_findings_node)
graph.add_edge(START, "classify")
graph.add_edge("classify", "load_skill")
graph.add_edge("load_skill", "research")
graph.add_conditional_edges(
    "research",
    tools_condition,
    {
        "tools": "tools",
        END: "summarize",
    },
)
graph.add_edge("tools", "research")
# graph.add_conditional_edges(
#     "tools", tools_condition, {"research": "research", "summarize": "summarize"}
# )
graph.add_edge("summarize", "chat")
graph.add_edge("chat", END)

# Use a production checkpointer backed by your database in the application.
workflow = graph.compile(checkpointer=MemorySaver())
workflow

In [ ]:
# Load this profile snapshot from your database; it remains authoritative.
birth_profile = {
    "user_id": "user_1234",
    "birth_profile_version": 1,
    "dob": "2002-02-08",
    "birth_time": "16:10",
    "birth_place": "Jaipur, Rajasthan, India",
}
thread_id = (
    f"{birth_profile['user_id']}:birth-profile:{birth_profile['birth_profile_version']}"
)

await workflow.ainvoke(
    {
        "question": HumanMessage(
            content="I am in Software Engineering field, when will I get a good job?"
        ),
        **birth_profile,
    },
    config={
        "configurable": {"thread_id": thread_id},
    },
)

In [ ]:
await workflow.ainvoke(
    {
        "question": HumanMessage(
            content="What is todays date? in short only date is needed"
        ),
        **birth_profile,
    },
    config={
        "configurable": {"thread_id": thread_id},
    },
)